# Notebook 4 — SHAP Explainability

**What this notebook does:** Explains the best model's predictions using SHAP
(SHapley Additive exPlanations) — which features drive churn risk overall, and
why individual customers were flagged as high-risk.

**Input:** `models/best_model.pkl`, `data/processed/X_test.csv`, `models/feature_names.pkl`, `models/scaler.pkl`

**Output:**
- Charts saved to `outputs/charts/shap/`
- `outputs/shap_feature_importance.csv` (used by notebook 7)

**Estimated run time:** ~1-2 minutes

In [1]:
# Imports and shared project utilities
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from utils import set_style, save_chart, load_csv_safely, load_joblib_safely, project_path

set_style()

In [2]:
# Load the best model and supporting artifacts from earlier notebooks
required_notebook_2 = "02_preprocessing.ipynb"
required_notebook_3 = "03_model_training.ipynb"

best_model = load_joblib_safely(project_path("models", "best_model.pkl"), required_notebook_3)
X_train = load_csv_safely(project_path("data", "processed", "X_train.csv"), required_notebook_2)
X_test = load_csv_safely(project_path("data", "processed", "X_test.csv"), required_notebook_2)
y_test = load_csv_safely(project_path("data", "processed", "y_test.csv"), required_notebook_2)["Churn"]
feature_names = load_joblib_safely(project_path("models", "feature_names.pkl"), required_notebook_2)
scaler = load_joblib_safely(project_path("models", "scaler.pkl"), required_notebook_2)

# X_test/X_train are already scaled (StandardScaler) since that's what the model was
# trained on. Inverse-transform a copy purely for human-readable printing/plotting —
# the SHAP values themselves are always computed on the scaled data the model actually sees.
X_test_original = pd.DataFrame(scaler.inverse_transform(X_test), columns=feature_names)

print(f"Best model type: {type(best_model).__name__}")
print(f"X_test shape: {X_test.shape}")

Best model type: LogisticRegression
X_test shape: (1407, 28)


## Create the SHAP explainer

The choice of explainer depends on the model type: tree-based models (Random Forest,
XGBoost, Gradient Boosting) use the fast, exact `TreeExplainer`; linear models
(Logistic Regression) use `LinearExplainer`. The result is normalised to a single
2D `Explanation` (one SHAP value per feature per customer, for the "Churn" class)
regardless of which explainer was used.

In [3]:
# Pick the right explainer for the model type that actually won in notebook 3
tree_based_models = ["RandomForestClassifier", "XGBClassifier", "GradientBoostingClassifier"]
model_type = type(best_model).__name__

if model_type in tree_based_models:
    print(f"{model_type} is tree-based -> using shap.TreeExplainer")
    explainer = shap.TreeExplainer(best_model)
else:
    print(f"{model_type} is not tree-based -> using shap.LinearExplainer")
    explainer = shap.LinearExplainer(best_model, X_train)

# Calculate SHAP values for the full test set
explanation = explainer(X_test)

# Tree explainers on binary classifiers return one set of SHAP values per class
# (shape: n_samples x n_features x 2) — keep only the "Churn" (positive) class
if explanation.values.ndim == 3:
    explanation = explanation[:, :, 1]

print("SHAP values shape:", explanation.values.shape)

Background dataset has 8260 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=8260 when initializing the masker.


LogisticRegression is not tree-based -> using shap.LinearExplainer
SHAP values shape: (1407, 28)


## Plot 1 — Summary Plot (beeswarm)

In [4]:
# Beeswarm plot: every dot is one customer, colour shows the feature's value
plt.figure(figsize=(12, 8))
shap.plots.beeswarm(explanation, max_display=15, show=False)
fig = plt.gcf()
fig.suptitle("SHAP Summary Plot — Top 15 Features", fontsize=14, fontweight="bold", y=1.02)
save_chart(fig, project_path("outputs", "charts", "shap", "shap_summary_beeswarm.png"))

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/shap/shap_summary_beeswarm.png (146.0KB)

## What this plot shows
Each dot is one customer. Red dots = feature value is high, blue = feature value is low.
Position on the x-axis shows whether that feature pushed the prediction towards churn
(positive) or away from churn (negative).

## Plot 2 — Bar Plot (global feature importance)

In [5]:
# Bar plot of mean absolute SHAP value per feature — overall importance ranking
plt.figure(figsize=(12, 8))
shap.plots.bar(explanation, max_display=15, show=False)
fig = plt.gcf()
fig.suptitle("SHAP Global Feature Importance — Top 15 Features", fontsize=14, fontweight="bold", y=1.02)
save_chart(fig, project_path("outputs", "charts", "shap", "shap_bar_global.png"))

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/shap/shap_bar_global.png (124.3KB)


## What this plot shows
Bars show the average magnitude of each feature's impact on the model's churn
prediction across all customers, ranked from most to least influential overall
(direction of the effect is not shown here — see the beeswarm plot above for that).

## Plot 3 — Waterfall Plot (single high-risk customer)

In [6]:
# Find the customer in X_test with the highest predicted churn probability
churn_probabilities = best_model.predict_proba(X_test)[:, 1]
high_risk_idx = int(np.argmax(churn_probabilities))
print(f"Highest-risk customer: row {high_risk_idx}, predicted churn probability = {churn_probabilities[high_risk_idx]:.3f}")

# Build a version of this customer's explanation with original-scale (unstandardised)
# feature values so the printed waterfall labels are human-readable
high_risk_explanation = shap.Explanation(
    values=explanation.values[high_risk_idx],
    base_values=explanation.base_values[high_risk_idx],
    data=X_test_original.iloc[high_risk_idx].values,
    feature_names=feature_names,
)

plt.figure(figsize=(12, 8))
shap.plots.waterfall(high_risk_explanation, show=False)
fig = plt.gcf()
fig.suptitle(f"Why Customer (row {high_risk_idx}) Was Flagged as High Risk", fontsize=13, fontweight="bold", y=1.02)
save_chart(fig, project_path("outputs", "charts", "shap", "shap_waterfall_high_risk.png"))

Highest-risk customer: row 667, predicted churn probability = 0.954


✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/shap/shap_waterfall_high_risk.png (113.3KB)


In [7]:
# Identify the top 3 features pushing this specific customer towards churn
customer_shap_values = pd.Series(explanation.values[high_risk_idx], index=feature_names)
customer_feature_values = X_test_original.iloc[high_risk_idx]

top_3_pushing_towards_churn = customer_shap_values.sort_values(ascending=False).head(3)

print("## Why the model flagged this customer\n")
print(f"Predicted churn probability: {churn_probabilities[high_risk_idx]:.1%}\n")
print("Top 3 features pushing towards churn:")
for feature, shap_value in top_3_pushing_towards_churn.items():
    raw_value = customer_feature_values[feature]
    display_value = round(raw_value, 1) if abs(raw_value - round(raw_value)) > 0.05 else int(round(raw_value))
    print(f"  - {feature} = {display_value}  (SHAP contribution: +{shap_value:.3f})")

## Why the model flagged this customer

Predicted churn probability: 95.4%

Top 3 features pushing towards churn:
  - MonthlyCharges = 95.6  (SHAP contribution: +4.314)
  - num_services = 4  (SHAP contribution: +2.418)
  - tenure = 1  (SHAP contribution: +1.557)


## Plot 4 — Dependence Plots for top 3 features

In [8]:
# Identify the top 3 features by mean absolute SHAP value across all customers
mean_abs_shap = np.abs(explanation.values).mean(axis=0)
top_3_feature_idx = np.argsort(mean_abs_shap)[::-1][:3]
top_3_features = [feature_names[i] for i in top_3_feature_idx]
print("Top 3 features by mean |SHAP value|:", top_3_features)

dependence_plot_explanations = {
    "tenure": "Newer customers (low tenure) are pushed towards churn, while long-tenured customers are pushed away from it — the relationship the retention team already suspected from the EDA.",
    "MonthlyCharges": "Higher monthly bills push customers towards churn, especially once bills exceed the typical range for their contract type.",
    "avg_monthly_spend": "Customers with a higher effective spend rate per month are more likely to churn, likely reflecting price sensitivity.",
    "TotalCharges": "Customers with low lifetime spend (usually newer customers) are pushed towards churn.",
    "is_long_term_contract": "Being on a long-term contract strongly pushes predictions away from churn.",
    "num_services": "Subscribing to more services generally pushes predictions away from churn, consistent with the bundling effect seen in the EDA.",
    "high_value_customer": "Being a high-value (above-median charge) customer pushes predictions towards churn, reflecting the premium/fiber pricing effect.",
    "StreamingMovies": "Subscribing to StreamingMovies is associated with lower churn, consistent with the general service-bundling retention effect.",
    "StreamingTV": "Subscribing to StreamingTV is associated with lower churn, consistent with the general service-bundling retention effect.",
    "PhoneService": "Having phone service is a weak but consistent retention signal, likely because it reflects a more bundled, embedded customer.",
    "OnlineSecurity": "Having OnlineSecurity is associated with materially lower churn — it is one of the stickiest add-ons in the bundle.",
    "OnlineBackup": "Having OnlineBackup is associated with lower churn as part of the broader service-bundling effect.",
    "DeviceProtection": "Having DeviceProtection is associated with lower churn as part of the broader service-bundling effect.",
    "TechSupport": "Having TechSupport is associated with materially lower churn, likely because it resolves the frustrations that would otherwise drive customers away.",
    "PaperlessBilling": "Paperless billing customers churn more, likely reflecting a more digitally self-served, less relationship-anchored customer base.",
    "SeniorCitizen": "Senior citizens are pushed towards churn more than non-seniors.",
}

for feature in top_3_features:
    plt.figure(figsize=(12, 6))
    shap.dependence_plot(
        feature,
        explanation.values,
        X_test_original,
        feature_names=feature_names,
        show=False,
    )
    fig = plt.gcf()
    fig.suptitle(f"SHAP Dependence Plot — {feature}", fontsize=13, fontweight="bold", y=1.02)
    save_chart(fig, project_path("outputs", "charts", "shap", f"shap_dependence_{feature}.png"))
    explanation_text = dependence_plot_explanations.get(
        feature, f"{feature} has a measurable, feature-value-dependent effect on churn risk that the retention team should investigate further."
    )
    print(f"Business takeaway ({feature}): {explanation_text}\n")


Top 3 features by mean |SHAP value|: ['num_services', 'MonthlyCharges', 'StreamingMovies']


✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/shap/shap_dependence_num_services.png (54.7KB)
Business takeaway (num_services): Subscribing to more services generally pushes predictions away from churn, consistent with the bundling effect seen in the EDA.

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/shap/shap_dependence_MonthlyCharges.png (83.0KB)
Business takeaway (MonthlyCharges): Higher monthly bills push customers towards churn, especially once bills exceed the typical range for their contract type.



✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/shap/shap_dependence_StreamingMovies.png (69.1KB)
Business takeaway (StreamingMovies): Subscribing to StreamingMovies is associated with lower churn, consistent with the general service-bundling retention effect.



## Plot 5 — Decision Plot for 5 customers

In [9]:
# Pick 2 high-risk (>0.7), 2 low-risk (<0.3) and 1 medium-risk (0.4-0.6) customer
rng = np.random.RandomState(42)

high_risk_pool = np.where(churn_probabilities > 0.7)[0]
low_risk_pool = np.where(churn_probabilities < 0.3)[0]
medium_risk_pool = np.where((churn_probabilities >= 0.4) & (churn_probabilities <= 0.6))[0]

selected_idx = np.concatenate([
    rng.choice(high_risk_pool, 2, replace=False),
    rng.choice(low_risk_pool, 2, replace=False),
    rng.choice(medium_risk_pool, 1, replace=False),
])

print("Selected customer rows (2 high, 2 low, 1 medium risk):", selected_idx)
print("Their churn probabilities:", np.round(churn_probabilities[selected_idx], 3))

plt.figure(figsize=(12, 8))
shap.decision_plot(
    explanation.base_values[selected_idx[0]],
    explanation.values[selected_idx],
    X_test_original.iloc[selected_idx],
    feature_names=feature_names,
    show=False,
)
fig = plt.gcf()
fig.suptitle("SHAP Decision Plot — 5 Customers (High / Medium / Low Risk)", fontsize=13, fontweight="bold", y=1.02)
save_chart(fig, project_path("outputs", "charts", "shap", "shap_decision_plot.png"))

Selected customer rows (2 high, 2 low, 1 medium risk): [ 211 1127  986  872 1323]
Their churn probabilities: [0.818 0.76  0.012 0.002 0.441]


✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/shap/shap_decision_plot.png (262.5KB)


## Business Summary Table

In [10]:
# Top 5 features by mean |SHAP value|, with the direction of their effect and a
# recommended business action for each
top_5_feature_idx = np.argsort(mean_abs_shap)[::-1][:5]
top_5_features = [feature_names[i] for i in top_5_feature_idx]

business_actions = {
    "tenure": ("Decreases churn as tenure rises", "Launch an early-life (0-12 month) onboarding and check-in program to retain new customers through their highest-risk period."),
    "MonthlyCharges": ("Increases churn as charges rise", "Review pricing for high-bill customers and consider loyalty discounts before they reach a churn-risk threshold."),
    "avg_monthly_spend": ("Increases churn as effective spend rises", "Flag customers with rapidly rising effective spend for proactive retention outreach."),
    "TotalCharges": ("Decreases churn as lifetime spend rises", "Prioritise retention offers for customers with low cumulative spend, who are disproportionately new and at risk."),
    "is_long_term_contract": ("Decreases churn when on a long-term contract", "Promote incentives (discounts, perks) for month-to-month customers to upgrade to 1- or 2-year contracts."),
    "num_services": ("Decreases churn as services subscribed increases", "Cross-sell add-on services (OnlineSecurity, TechSupport) to single-service customers to increase stickiness."),
    "high_value_customer": ("Increases churn for above-median charge customers", "Create a premium retention tier with added perks for high-value customers to protect this high-revenue segment."),
    "PaperlessBilling": ("Increases churn for paperless billing customers", "Investigate whether paperless billing correlates with a more price-sensitive, digitally self-served (and less loyal) customer segment."),
    "SeniorCitizen": ("Increases churn for senior citizens", "Offer simplified plans and accessible customer support tailored to senior customers."),
    "StreamingMovies": ("Decreases churn when subscribed", "Bundle StreamingMovies into entry-level plans as a low-cost retention lever."),
    "StreamingTV": ("Decreases churn when subscribed", "Bundle StreamingTV into entry-level plans as a low-cost retention lever."),
    "PhoneService": ("Decreases churn when subscribed", "Encourage single-service internet customers to add phone service as part of a bundle discount."),
    "OnlineSecurity": ("Decreases churn when subscribed", "Proactively offer OnlineSecurity free for the first 3 months to at-risk customers — it is one of the stickiest add-ons."),
    "OnlineBackup": ("Decreases churn when subscribed", "Cross-sell OnlineBackup to customers with 0-2 services as an entry point into the bundle."),
    "DeviceProtection": ("Decreases churn when subscribed", "Cross-sell DeviceProtection to customers with 0-2 services as an entry point into the bundle."),
    "TechSupport": ("Decreases churn when subscribed", "Offer a free trial of TechSupport to high-risk customers flagged by the model — it directly addresses service frustration."),
}

rows = []
for rank, feature in enumerate(top_5_features, start=1):
    effect, action = business_actions.get(
        feature,
        ("Measurable effect on churn risk", "Investigate this feature further with the retention team to design a targeted intervention.")
    )
    rows.append({"Rank": rank, "Feature": feature, "Effect on Churn": effect, "Business Action": action})

summary_df = pd.DataFrame(rows)

print("| Rank | Feature | Effect on Churn | Business Action |")
print("|------|---------|-----------------|-----------------|")
for _, row in summary_df.iterrows():
    rank_val = row["Rank"]
    feature_val = row["Feature"]
    effect_val = row["Effect on Churn"]
    action_val = row["Business Action"]
    print(f"| {rank_val} | {feature_val} | {effect_val} | {action_val} |")


| Rank | Feature | Effect on Churn | Business Action |
|------|---------|-----------------|-----------------|
| 1 | num_services | Decreases churn as services subscribed increases | Cross-sell add-on services (OnlineSecurity, TechSupport) to single-service customers to increase stickiness. |
| 2 | MonthlyCharges | Increases churn as charges rise | Review pricing for high-bill customers and consider loyalty discounts before they reach a churn-risk threshold. |
| 3 | StreamingMovies | Decreases churn when subscribed | Bundle StreamingMovies into entry-level plans as a low-cost retention lever. |
| 4 | StreamingTV | Decreases churn when subscribed | Bundle StreamingTV into entry-level plans as a low-cost retention lever. |
| 5 | PhoneService | Decreases churn when subscribed | Encourage single-service internet customers to add phone service as part of a bundle discount. |


## Save full feature importance table (used by notebook 7 for the Power BI export)

In [11]:
from utils import save_dataframe

# Save the full ranked feature importance table so notebook 7 can build the
# Power BI feature_importance.csv export without recomputing SHAP values
full_feature_importance = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap_value": mean_abs_shap,
}).sort_values("mean_abs_shap_value", ascending=False).reset_index(drop=True)

save_dataframe(full_feature_importance, project_path("outputs", "shap_feature_importance.csv"))


✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/shap_feature_importance.csv (1.0KB) — 28 rows, 2 columns
